In [1]:
import pandas as pd # For DataFrames, Series, and reading csv data in.
import seaborn as sns # Graphing, built ontop of MatPlot for ease-of-use and nicer diagrams.
import matplotlib.pyplot as plt # MatPlotLib for graphing data visually. Seaborn more likely to be used.
import numpy as np # For manipulating arrays and changing data into correct formats for certain libraries
from sklearn.preprocessing import StandardScaler  # For Normalization
import scikitplot # Confusion matrix plotting

In [2]:
df1=pd.read_csv('data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
df2 = pd.read_csv('data/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv')
df3 = pd.read_csv('data/Friday-WorkingHours-Morning.pcap_ISCX.csv')
df4 = pd.read_csv('data/Monday-WorkingHours.pcap_ISCX.csv')
df5 = pd.read_csv('data/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv')
df6 = pd.read_csv('data/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv')
df7 = pd.read_csv('data/Tuesday-WorkingHours.pcap_ISCX.csv')
df8 = pd.read_csv('data/Wednesday-workingHours.pcap_ISCX.csv')


In [3]:
df = pd.concat([df1, df2], axis=0)
del df1, df2
df = pd.concat([df, df3], axis=0)
del df3
df = pd.concat([df, df4], axis=0)
del df4
df = pd.concat([df, df5], axis=0)
del df5
df = pd.concat([df, df6], axis=0)
del df6
df = pd.concat([df, df7], axis=0)
del df7
df = pd.concat([df, df8], axis=0)
del df8

df.shape

(2830743, 79)

In [4]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [5]:
df.dtypes

 Destination Port                int64
 Flow Duration                   int64
 Total Fwd Packets               int64
 Total Backward Packets          int64
Total Length of Fwd Packets      int64
                                ...   
Idle Mean                      float64
 Idle Std                      float64
 Idle Max                        int64
 Idle Min                        int64
 Label                          object
Length: 79, dtype: object

In [6]:
#clean column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
df.head()

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [7]:
df.dtypes


destination_port                 int64
flow_duration                    int64
total_fwd_packets                int64
total_backward_packets           int64
total_length_of_fwd_packets      int64
                                ...   
idle_mean                      float64
idle_std                       float64
idle_max                         int64
idle_min                         int64
label                           object
Length: 79, dtype: object

In [8]:
df_cleaned = df.copy()
#.reset_index() method resets the Pandas Dataframe indexes, for the rows. Useful to do after merging rows, as this messes up the indexes.
df_cleaned = df_cleaned.reset_index()
df_cleaned.drop('index', axis=1, inplace=True)
df_cleaned.dtypes

destination_port                 int64
flow_duration                    int64
total_fwd_packets                int64
total_backward_packets           int64
total_length_of_fwd_packets      int64
                                ...   
idle_mean                      float64
idle_std                       float64
idle_max                         int64
idle_min                         int64
label                           object
Length: 79, dtype: object

In [10]:
# Saving the label attribute to df_labels before dropping it.
df_labels = df_cleaned['label']
df_labels.unique()

array(['BENIGN', 'DDoS', 'PortScan', 'Bot', 'Infiltration',
       'Web Attack � Brute Force', 'Web Attack � XSS',
       'Web Attack � Sql Injection', 'FTP-Patator', 'SSH-Patator',
       'DoS slowloris', 'DoS Slowhttptest', 'DoS Hulk', 'DoS GoldenEye',
       'Heartbleed'], dtype=object)

In [11]:
#performing one-hot encoding
y_ohe=pd.get_dummies(df_labels,dtype=int)
y_ohe.head()

,BENIGN,Bot,DDoS,DoS GoldenEye,DoS Hulk,DoS Slowhttptest,DoS slowloris,FTP-Patator,Heartbleed,Infiltration,PortScan,SSH-Patator,Web Attack � Brute Force,Web Attack � Sql Injection,Web Attack � XSS
0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Run drop wala code only code 

In [12]:
#dropping original label column
#df_cleaned.drop('label',axis=1,inplace=True) #run only once
df_cleaned.head()

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,1,20,0.0,0.0,0,0,0.0,0.0,0,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,0,20,0.0,0.0,0,0,0.0,0.0,0,0
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,1,20,0.0,0.0,0,0,0.0,0.0,0,0


In [13]:
#concatenating
df_cleaned=pd.concat([df_cleaned,y_ohe],axis=1)

In [13]:
df_cleaned.head()

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,DoS Slowhttptest,DoS slowloris,FTP-Patator,Heartbleed,Infiltration,PortScan,SSH-Patator,Web Attack � Brute Force,Web Attack � Sql Injection,Web Attack � XSS
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,0,0,0,0,0,0,0,0,0,0
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,0,0,0,0,0,0,0,0,0,0
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
#clean data is now ready
df_cleaned.shape

(2830743, 93)

In [20]:
#handling nan and inf values
def clean_dataset(df_cleaned):
    assert isinstance(df_cleaned, pd.DataFrame), "df needs to be a pd.DataFrame"
    df_cleaned.dropna(inplace=True)
    indices_to_keep = ~df_cleaned.isin([np.nan, np.inf, -np.inf]).any(axis=1)
    return df_cleaned[indices_to_keep].astype(np.float64)

df_cleaned = clean_dataset(df_cleaned)

In [21]:
df_cleaned.columns.get_loc("BENIGN")

78

In [22]:
#segregating features and target variables #pandas has been inconsistent with negative indexing so we are going with positive indexing only
X=df_cleaned.iloc[:,:78]
y_target=df_cleaned.iloc[:,78:94]

In [23]:
#dividing train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_target, test_size=0.2, random_state=114)



In [24]:
#saving them, make sure to include index=False for correct output, else inherits an unwanted (unnecessary) index column as 0th column
X_train.to_csv('X_train_data.csv', index=False) 
X_test.to_csv('X_test_data.csv', index=False)
y_train.to_csv('y_train_data.csv', index=False)
y_test.to_csv('y_test_data.csv', index=False)

Beginning of Feature scaling using Z-score Scaling (Standardization), it should always be done after Splitting to hide mean,sd of test dataset


In [2]:
sc=StandardScaler()
#calculates the mean and standard deviation from the training set only (assuming data has no pratiular distribution)
# and then uses these statistics to standardize both the training set and test set
df_train=pd.read_csv('eda1_results_before_scaling/X_train_data.csv')
df_train.head() #before scaling

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min
0,443.0,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.00000,...,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,88.0,97.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.00000,...,0.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,54146.0,3.0,2.0,0.0,31.0,0.0,31.0,0.0,15.5,21.92031,...,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,53.0,24367.0,2.0,2.0,66.0,270.0,33.0,33.0,33.0,0.00000,...,1.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,53.0,31067.0,2.0,2.0,80.0,336.0,40.0,40.0,40.0,0.00000,...,1.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
#df_train = df_train.reset_index()
#leaving port number as it is because it may be relevant 
df_train.iloc[:,1:]=sc.fit_transform(df_train.iloc[:,1:]) #Must have equal len keys and value when setting with an ndarray
df_train.head() #after scaling

,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,act_data_pkt_fwd,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min
0,443.0,-0.439950,-0.009886,-0.010480,-0.081001,-0.007206,-0.281352,-0.210949,-0.280741,-0.245450,...,-0.006922,0.002386,-0.125867,-0.106047,-0.150001,-0.100995,-0.352468,-0.109642,-0.357413,-0.339497
1,88.0,-0.439947,-0.011221,-0.006476,-0.082817,-0.007206,-0.289720,-0.310405,-0.312979,-0.245450,...,-0.008490,0.002398,-0.125867,-0.106047,-0.150001,-0.100995,-0.352468,-0.109642,-0.357413,-0.339497
2,54146.0,-0.439950,-0.009886,-0.010480,-0.078125,-0.007206,-0.246484,-0.310405,-0.229696,-0.167398,...,-0.008490,0.002386,-0.125867,-0.106047,-0.150001,-0.100995,-0.352468,-0.109642,-0.357413,-0.339497
3,53.0,-0.439227,-0.009886,-0.008478,-0.072827,-0.007086,-0.243695,0.236603,-0.135667,-0.245450,...,-0.006922,0.002386,-0.125867,-0.106047,-0.150001,-0.100995,-0.352468,-0.109642,-0.357413,-0.339497
4,53.0,-0.439028,-0.009886,-0.008478,-0.070707,-0.007057,-0.233932,0.352635,-0.098056,-0.245450,...,-0.006922,0.002398,-0.125867,-0.106047,-0.150001,-0.100995,-0.352468,-0.109642,-0.357413,-0.339497


In [11]:
df_train.to_csv('X_train_scaled_data.csv', index=False)


In [13]:
del df_train

In [14]:
df_test=pd.read_csv('eda1_results_before_scaling/X_test_data.csv')

In [15]:
df_test.iloc[:,1:]=sc.transform(df_test.iloc[:,1:])

In [16]:
df_test.to_csv('X_test_scaled_data.csv', index=False)

del df_test

End of Feature Scaling